# Persona Vectors: Extracting Personality from Gemma 4

Based on Anthropic Research (Aug 2025) — "Persona Vectors: Personality in the Weights"
Adapted for Google Gemma 4 E4B-it

### What this notebook does:
1. Extract persona vectors by comparing activations between opposing behavioral prompts
2. Visualize persona vectors across all 42 layers
3. Steer model behavior by injecting persona vectors during inference
4. Compare steered vs unsteered generation across 4 personality dimensions

In [ ]:
%matplotlib inline

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os
import textwrap
from transformers import AutoProcessor, AutoModelForCausalLM
from torch.nn.functional import cosine_similarity as cos_sim

# Configuration
MODEL_ID = "google/gemma-4-E4B-it"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PyTorch version: {}".format(torch.__version__))
print("CUDA available: {}".format(torch.cuda.is_available()))
if torch.cuda.is_available():
    print("GPU: {}".format(torch.cuda.get_device_name(0)))

## Step 1: Load Model

Load Gemma 4 E4B-it with `dtype="auto"` and `device_map="auto"`. Uses `AutoProcessor` instead of `AutoTokenizer`.

In [ ]:
print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    dtype="auto",
    device_map="auto",
)
model.eval()
print("Model loaded successfully")
print("Number of layers: {}".format(model.config.text_config.num_hidden_layers))
print("Hidden size: {}".format(model.config.text_config.hidden_size))

## Step 2: Define Contrastive Prompt Pairs

Persona vectors are extracted by comparing activations between **opposing behavioral prompts**. For each personality dimension, we define prompts that push the model toward opposite ends of a behavioral spectrum. The difference in mean activations yields a direction in activation space that encodes that personality trait.

In [ ]:
PERSONA_DIMENSIONS = {
    "helpfulness": {
        "positive": [
            "You are an extremely helpful assistant who goes above and beyond to help users.",
            "You always try your best to provide thorough, detailed, and useful answers.",
            "You are eager to help and provide comprehensive assistance.",
        ],
        "negative": [
            "You are an unhelpful assistant who gives minimal, dismissive responses.",
            "You don't care about helping and provide the bare minimum.",
            "You are reluctant to help and give vague, incomplete answers.",
        ],
    },
    "formality": {
        "positive": [
            "You are a formal, professional assistant who uses precise academic language.",
            "You communicate with utmost professionalism and formality.",
            "You always use formal language, proper grammar, and professional tone.",
        ],
        "negative": [
            "You are a casual, laid-back assistant who talks like a friend.",
            "You use slang, informal language, and a relaxed conversational tone.",
            "You chat casually, use contractions, and keep things super chill.",
        ],
    },
    "confidence": {
        "positive": [
            "You are extremely confident in your knowledge and express certainty.",
            "You state facts with authority and conviction.",
            "You are decisive and assertive in all your responses.",
        ],
        "negative": [
            "You are uncertain and hedge every statement with qualifiers.",
            "You express doubt about everything and use lots of 'maybe' and 'perhaps'.",
            "You are hesitant and unsure, always questioning your own answers.",
        ],
    },
    "verbosity": {
        "positive": [
            "You give extremely detailed, thorough, long-form responses.",
            "You elaborate extensively on every point with examples and explanations.",
            "You provide comprehensive answers that cover every angle.",
        ],
        "negative": [
            "You give extremely brief, concise, minimal responses.",
            "You use as few words as possible to answer.",
            "You are terse and to the point, never elaborating.",
        ],
    },
}

print("Defined {} persona dimensions:".format(len(PERSONA_DIMENSIONS)))
for dim_name, prompts in PERSONA_DIMENSIONS.items():
    print("  {} — {} positive, {} negative prompts".format(
        dim_name, len(prompts["positive"]), len(prompts["negative"])
    ))

## Step 3: Extract Persona Vectors

For each persona dimension, we:
1. Run all **positive** system prompts through the model and capture hidden states at every layer using forward hooks
2. Run all **negative** system prompts and capture hidden states
3. Compute the **mean activation** for positive and negative sets
4. The **persona vector** = mean(positive) - mean(negative) at each layer

This contrastive approach isolates the direction in activation space that corresponds to that personality trait, cancelling out content that is shared between both polarities.

In [ ]:
def extract_persona_vectors(model, processor, persona_dims, target_layers=None):
    """Extract persona vectors using contrastive activation differences."""
    num_layers = model.config.text_config.num_hidden_layers
    if target_layers is None:
        target_layers = list(range(num_layers))

    persona_vectors = {}

    for dim_name, prompts in persona_dims.items():
        print("Extracting: {}".format(dim_name))
        pos_activations = {l: [] for l in target_layers}
        neg_activations = {l: [] for l in target_layers}

        for polarity, prompt_list in [("positive", prompts["positive"]), ("negative", prompts["negative"])]:
            for prompt in prompt_list:
                # Use chat template with system prompt
                messages = [
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": "Hello, how are you?"},
                ]
                text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=text, return_tensors="pt").to(model.device)

                # Capture hidden states via hooks
                hidden_states = {}
                handles = []
                for layer_idx in target_layers:
                    def make_hook(idx):
                        def hook(module, input, output):
                            hidden_states[idx] = output[0].detach().cpu().float()
                        return hook
                    h = model.model.language_model.layers[layer_idx].register_forward_hook(make_hook(layer_idx))
                    handles.append(h)

                with torch.no_grad():
                    model(**inputs)

                for h in handles:
                    h.remove()

                # Store mean activation across sequence positions -> [hidden_dim]
                target = pos_activations if polarity == "positive" else neg_activations
                for idx in target_layers:
                    h = hidden_states[idx]
                    if h.dim() == 3:
                        h = h.mean(dim=1).squeeze(0)  # [batch, seq, hidden] -> [hidden]
                    elif h.dim() == 2:
                        h = h.mean(dim=0)  # [seq, hidden] -> [hidden]
                    target[idx].append(h)

        # Compute persona vector: mean(positive) - mean(negative)
        vectors = {}
        for idx in target_layers:
            pos_mean = torch.stack(pos_activations[idx]).mean(dim=0)
            neg_mean = torch.stack(neg_activations[idx]).mean(dim=0)
            vectors[idx] = (pos_mean - neg_mean).squeeze(0)  # [hidden_dim]

        persona_vectors[dim_name] = vectors
        print("  Done: {} vectors across {} layers".format(dim_name, len(target_layers)))

    return persona_vectors

persona_vectors = extract_persona_vectors(model, processor, PERSONA_DIMENSIONS)
print("\nExtraction complete! Dimensions: {}".format(list(persona_vectors.keys())))

## Step 4: Visualize Persona Vector Norms

The L2 norm shows WHERE in the network each personality trait is most strongly encoded across all 42 Gemma 4 layers.

In [ ]:
num_layers = model.config.text_config.num_hidden_layers
layer_indices = list(range(num_layers))

dim_colors = {
    "helpfulness": "#2196F3",
    "formality": "#FF9800",
    "confidence": "#4CAF50",
    "verbosity": "#9C27B0",
}
dim_markers = {
    "helpfulness": "o",
    "formality": "s",
    "confidence": "^",
    "verbosity": "D",
}

fig, ax = plt.subplots(figsize=(14, 6))

strongest_layers = {}  # Track which layer has peak norm per dimension

for dim_name in PERSONA_DIMENSIONS:
    norms = []
    for layer_idx in layer_indices:
        norm = torch.norm(persona_vectors[dim_name][layer_idx], p=2).item()
        norms.append(norm)
    strongest_layers[dim_name] = int(np.argmax(norms))
    ax.plot(
        layer_indices, norms,
        marker=dim_markers[dim_name],
        color=dim_colors[dim_name],
        label="{} (peak: layer {})".format(dim_name, strongest_layers[dim_name]),
        linewidth=2, markersize=4, alpha=0.85,
    )

ax.set_xlabel("Layer", fontsize=13)
ax.set_ylabel("L2 Norm of Persona Vector", fontsize=13)
ax.set_title("Gemma 4 — Persona Vector Strength Across 42 Layers", fontsize=15, fontweight="bold")
ax.legend(fontsize=11, loc="best")
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, num_layers - 0.5)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_06_persona_norms.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
print("Strongest layers: {}".format(strongest_layers))
plt.show()

## Step 5: Persona Similarity Heatmap

Cosine similarity between persona vectors at their strongest layers reveals which personality traits are correlated or anti-correlated in Gemma 4's representations.

In [ ]:
dim_names = list(PERSONA_DIMENSIONS.keys())
n_dims = len(dim_names)

# Use the strongest layer for each dimension
sim_matrix = np.zeros((n_dims, n_dims))
for i, dim_a in enumerate(dim_names):
    for j, dim_b in enumerate(dim_names):
        layer_a = strongest_layers[dim_a]
        layer_b = strongest_layers[dim_b]
        # Use layer of the row dimension for cross-comparison
        layer_idx = layer_a if i <= j else layer_b
        vec_a = persona_vectors[dim_a][layer_idx]
        vec_b = persona_vectors[dim_b][layer_idx]
        sim = cos_sim(vec_a.unsqueeze(0), vec_b.unsqueeze(0)).item()
        sim_matrix[i, j] = sim

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")

# Add text annotations
for i in range(n_dims):
    for j in range(n_dims):
        color = "white" if abs(sim_matrix[i, j]) > 0.6 else "black"
        ax.text(j, i, "{:.2f}".format(sim_matrix[i, j]),
                ha="center", va="center", fontsize=13, fontweight="bold", color=color)

ax.set_xticks(range(n_dims))
ax.set_yticks(range(n_dims))
ax.set_xticklabels(dim_names, fontsize=12, rotation=45, ha="right")
ax.set_yticklabels(dim_names, fontsize=12)
ax.set_title("Gemma 4 — Inter-Persona Cosine Similarity (Strongest Layer)", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8, label="Cosine Similarity")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_07_persona_similarity.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Step 6: Steer Model Behavior

Now we use the extracted persona vectors to **steer** the model's behavior at inference time. By adding (or subtracting) a persona vector to the residual stream at a specific layer, we can push the model's outputs toward or away from a personality trait — without any fine-tuning or prompt engineering.

In [ ]:
def generate_with_steering(model, processor, prompt, persona_vector, layer_idx, alpha=1.0, max_new_tokens=100):
    """Generate text with persona vector steering.

    Args:
        model: The language model
        processor: The processor (handles tokenization and chat template)
        prompt: User prompt text
        persona_vector: The persona vector to inject (1D tensor, hidden_dim)
        layer_idx: Which layer to inject the steering vector at
        alpha: Scaling factor (positive = toward trait, negative = away from trait)
        max_new_tokens: Maximum tokens to generate

    Returns:
        Generated text string
    """
    def steering_hook(module, input, output):
        # In-place modification to avoid tuple/ModelOutput return issues
        hidden = output[0]
        steering = persona_vector.to(hidden.device).to(hidden.dtype)
        if hidden.dim() == 3:
            hidden.data.add_(alpha * steering.unsqueeze(0).unsqueeze(0))
        elif hidden.dim() == 2:
            hidden.data.add_(alpha * steering.unsqueeze(0))
        # No return — in-place modification

    handle = model.model.language_model.layers[layer_idx].register_forward_hook(steering_hook)

    messages = [{"role": "user", "content": prompt}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    handle.remove()
    return processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# --- Steering experiment: helpfulness at different alpha values ---
test_prompt = "Can you explain what machine learning is?"
dim_name = "helpfulness"
best_layer = strongest_layers[dim_name]
vec = persona_vectors[dim_name][best_layer]

alphas = [-2.0, -1.0, 0.0, 1.0, 2.0]
print("Steering '{}' at layer {} with different alpha values".format(dim_name, best_layer))
print("Prompt: '{}'\n".format(test_prompt))
print("=" * 80)

steering_results = {}
for alpha in alphas:
    if alpha == 0.0:
        # No steering — baseline generation
        messages = [{"role": "user", "content": test_prompt}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        result = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    else:
        result = generate_with_steering(model, processor, test_prompt, vec, best_layer, alpha=alpha, max_new_tokens=100)

    steering_results[alpha] = result
    print("\n--- alpha = {} ---".format(alpha))
    print(result)
    print("-" * 80)

## Step 7: Compare All Dimensions

Generate steered outputs for all four persona dimensions and visualize them side by side — negative vs positive steering.

In [ ]:
comparison_prompt = "What is artificial intelligence?"
steering_alpha = 1.5

all_steered = {}
for dim_name in PERSONA_DIMENSIONS:
    best_layer = strongest_layers[dim_name]
    vec = persona_vectors[dim_name][best_layer]

    neg_text = generate_with_steering(
        model, processor, comparison_prompt, vec, best_layer,
        alpha=-steering_alpha, max_new_tokens=120
    )
    pos_text = generate_with_steering(
        model, processor, comparison_prompt, vec, best_layer,
        alpha=steering_alpha, max_new_tokens=120
    )
    all_steered[dim_name] = {"negative": neg_text, "positive": pos_text}
    print("Generated steered outputs for: {}".format(dim_name))

# Build the comparison figure
fig, axes = plt.subplots(len(PERSONA_DIMENSIONS), 2, figsize=(18, 5 * len(PERSONA_DIMENSIONS)))

for row, dim_name in enumerate(PERSONA_DIMENSIONS):
    for col, polarity in enumerate(["negative", "positive"]):
        ax = axes[row, col]
        ax.axis("off")

        text = all_steered[dim_name][polarity]
        wrapped = textwrap.fill(text, width=70)

        if col == 0:
            title = "{}\nalpha = -{:.1f}".format(dim_name.upper(), steering_alpha)
            title_color = "#D32F2F"
        else:
            title = "{}\nalpha = +{:.1f}".format(dim_name.upper(), steering_alpha)
            title_color = "#388E3C"

        ax.set_title(title, fontsize=13, fontweight="bold", color=title_color, loc="left")
        ax.text(
            0.02, 0.92, wrapped,
            transform=ax.transAxes, fontsize=9,
            verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8),
            wrap=True,
        )

fig.suptitle(
    "Gemma 4 — Persona Steering: Negative vs Positive (alpha={})".format(steering_alpha),
    fontsize=16, fontweight="bold", y=1.01
)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_08_steering_comparison.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Step 8: Top Dimensions of Persona Vectors

For each persona dimension at its strongest layer, show the top-20 most activated hidden dimensions. This reveals which specific neurons encode personality traits.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, dim_name in enumerate(PERSONA_DIMENSIONS):
    ax = axes[idx // 2, idx % 2]
    best_layer = strongest_layers[dim_name]
    vec = persona_vectors[dim_name][best_layer].numpy()

    # Get top-20 dimensions by absolute value
    abs_vals = np.abs(vec)
    top_indices = np.argsort(abs_vals)[-20:][::-1]
    top_values = vec[top_indices]

    colors = ["#388E3C" if v > 0 else "#D32F2F" for v in top_values]
    ax.bar(range(20), top_values, color=colors, alpha=0.8, edgecolor="white", linewidth=0.5)

    ax.set_xticks(range(20))
    ax.set_xticklabels(["d{}".format(i) for i in top_indices], fontsize=7, rotation=45, ha="right")
    ax.set_ylabel("Activation Difference", fontsize=11)
    ax.set_title("{} (layer {})".format(dim_name.capitalize(), best_layer), fontsize=13, fontweight="bold")
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="-")
    ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Gemma 4 — Top-20 Active Dimensions per Persona Vector", fontsize=15, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_09_persona_top_dims.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Summary and Key Findings

### What we observed:

1. **Persona vectors have layer-specific strength**: The L2 norm plot shows that persona vectors are not uniformly distributed across Gemma 4's 42 layers. Certain layers encode personality traits much more strongly than others, typically in the middle-to-late layers where the model transitions from syntactic to semantic processing.

2. **Personality traits have distinct representations**: The cosine similarity heatmap reveals the degree of overlap between different persona dimensions. Some traits (e.g., helpfulness and verbosity) may share activation directions, while others (e.g., formality and confidence) may be more independent.

3. **Steering works without fine-tuning**: By simply adding a scaled persona vector to the residual stream at the right layer, we can measurably shift the model's behavioral style. Positive alpha pushes toward the trait; negative alpha pushes away from it.

4. **Sparse encoding**: The top-20 dimension plots show that persona vectors are relatively sparse — a small number of hidden dimensions carry most of the personality signal, consistent with Anthropic's finding that personality is encoded in identifiable, low-dimensional subspaces.

### Gemma 4 vs Llama 3.1:
- Gemma 4 has 42 layers vs Llama's 32, providing more granularity in where persona vectors peak
- Hybrid sliding window + global attention may affect how personality traits distribute across layers
- Both architectures support in-place steering via forward hooks on `model.model.layers[i]`

### Methodology notes:
- Persona vectors are computed as the **mean difference** between positive and negative contrastive prompts
- We average across sequence positions to get a single vector per layer
- Steering is applied as an additive in-place intervention to the residual stream
- All inference uses greedy decoding (`do_sample=False`) for reproducibility

In [ ]:
# Clean up and summary statistics
print("=" * 60)
print("PERSONA VECTOR EXTRACTION — SUMMARY")
print("=" * 60)
print("Model: {}".format(MODEL_ID))
print("Total layers: {}".format(model.config.text_config.num_hidden_layers))
print("Hidden size: {}".format(model.config.text_config.hidden_size))
print("Persona dimensions: {}".format(len(PERSONA_DIMENSIONS)))
print("Prompts per polarity: {}".format(len(list(PERSONA_DIMENSIONS.values())[0]["positive"])))
print()

print("Strongest layers per dimension:")
for dim_name in PERSONA_DIMENSIONS:
    best_layer = strongest_layers[dim_name]
    norm = torch.norm(persona_vectors[dim_name][best_layer], p=2).item()
    print("  {:<15s} layer {:>2d}  (norm: {:.4f})".format(dim_name, best_layer, norm))

print()
print("Charts saved to: {}".format(OUTPUT_DIR))
print("  g4_06_persona_norms.png")
print("  g4_07_persona_similarity.png")
print("  g4_08_steering_comparison.png")
print("  g4_09_persona_top_dims.png")
print()

# Free GPU memory
del persona_vectors
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("GPU memory released.")